# Train and test I.A for model select

### Import and create itens for train

In [ ]:
import os 
from time import sleep
import torch
import torch.nn as nn
import torch.optim as optim
from tqdm import tqdm
from utils.trainImplematation import TrainDataset
from utils.metricsIaLearnig import Metrics
from utils.models import ImagemDetectionModels

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")   

print("Dispositivo utilizado: ", device)

In [ ]:
dataSetPath = "../imagens"    

image_size = (224, 224)
batch_size = 32
dataSet = TrainDataset(dataSetPath, image_size, batch_size)
trainData = dataSet.loadTrainData()
testData = dataSet.loadTestData()


In [ ]:
import os
import torch
from time import sleep
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
num_classes = 2
num_epochs = 100

models = [
    {
        "model": ImagemDetectionModels.resnet101(num_classes),
        "name": "ResNet101"
    },
    {
        "model": ImagemDetectionModels.vgg16(num_classes),
        "name": "VGG16"
    },
    {
        "model": ImagemDetectionModels.vgg16_bn(num_classes),
        "name": "VGG16_BN"
    },
]

for m in models:
    model = m["model"]
    name = m["name"]
    
    metrics = Metrics(model=model, device=device, testData=testData, saveFig=True, nameModel=name)

    try:
        model = model.to(device)
        model.train()
    except Exception as e:
        continue

    criterion = torch.nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

    for epoch in range(1, num_epochs + 1):
        model.train()
        pbar_batch = tqdm(trainData, unit="batch", desc=f"{name} - Epoch {epoch}")
        losses = []

        for images, labels in pbar_batch:
            images, labels = images.to(device), labels.to(device)
            scores = model(images)
            loss = criterion(scores, labels)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            losses.append(loss.item())
            avg_loss = sum(losses) / len(losses)
            pbar_batch.set_postfix(loss=f"{avg_loss:.4f}")

        metrics.trainTestData()

        wait_time = 0
        while any(v is None for v in [metrics.yPred, metrics.yTrue, metrics.yScore]) and wait_time < 10:
            sleep(1)
            wait_time += 1

        if wait_time >= 10:
            continue

        metrics.showAll()
        sleep(0.1)

    model_save_path = f"weights/{name.lower()}"
    os.makedirs(model_save_path, exist_ok=True)
    model_path = os.path.join(model_save_path, f"{name.lower()}_model_checkpoint.pth")
    torch.save(model.state_dict(), model_path)


In [ ]:
'''
    saves the trained model weights to disk, creating the directory 
    if it does not already exist.
'''
model_save_path = "weights/resnet"
if not os.path.exists(model_save_path):
    os.makedirs(model_save_path)

torch.save(model.state_dict(), model_save_path + "/resnet_model_checkpoint.pth")